# Data Manipulation

A photograph of a shirt reaches a model as a grid of numbers, one per
pixel, and everything else the model touches is numbers too: its parameters,
the values it computes along the way, and the gradients used to train it. A
deep learning program spends nearly all of its time computing with such
collections of numbers. How should we store them, and how can we compute
with them quickly?

Let us take one humble computation, the rescaling of images, to see what is
at stake. Training a model processes its examples in small groups called
*minibatches*, and a minibatch of 256 such photographs, grayscale and of
$28 \times 28$ pixels each, holds $256 \times 28 \times 28 = 200{,}704$
numbers. Image files store each pixel as an integer from 0 to 255, and
before a model sees the images we rescale every pixel to a real number
between 0 and 1 by dividing it by 255. In plain Python, this takes three
nested loops over the images, the rows, and the columns, and the interpreter
carries out the 200,704 divisions one at a time. In deep learning code it
takes one expression, `images / 255`, which the library applies to the whole
minibatch at once in compiled code, on a CPU or a GPU. This is how deep
learning code computes throughout: with whole arrays, and hardly ever with
loops over their elements.

An array of numbers arranged along any number of axes is called a *tensor*.
If you know NumPy [@Harris.Millman.Walt.ea.2020], much of this section
will look familiar, and you may read it quickly. The tensor classes of deep
learning libraries (`ndarray` in MXNet, `Tensor` in PyTorch and TensorFlow,
`jax.Array` in JAX) resemble NumPy's `ndarray`, with two additions, and it
is these that make deep learning practical. They run their operations on
GPUs and other accelerators as well as on CPUs, and the libraries can
compute derivatives of computations on them automatically
(that section).

This section shows how to create tensors and inspect their shape and type,
how to rearrange, select, and change their elements, and how to compute with
whole tensors at once, including tensors whose shapes differ. It ends with
what these computations cost in memory and how to convert tensors to other
Python objects. The minibatch of images is our running example. Where a
tensor lives, in main memory or on a GPU, changes no result in this chapter;
a later chapter shows how to choose.

We import `jax` and its NumPy-like interface `jax.numpy`, by convention
under the alias `jnp`.

In [ ]:
import random
import jax
from jax import numpy as jnp

## Tensor Creation

### Shape

A tensor arranges its numbers along its *axes*. A vector has one axis and a
matrix has two. The minibatch of images has four: one for the image, one for
the color channel (a grayscale image has a single channel), and one each for
the rows and the columns of pixels. Axes are numbered from 0, so for the
minibatch, axis 0 indexes the images and axis 3 the columns. The number of
axes is the *order* of a tensor (`ndim` in code), and the minibatch has
order 4.

The simplest way to create a tensor is `arange(n)`, which returns a vector of
the integers from 0 up to, but not including, $n$. We ask for 32-bit
floating-point numbers, the type that we compute with throughout:

In [ ]:
x = jnp.arange(12, dtype=jnp.float32)
x

The numbers in a tensor are its *elements*, and the number of elements is
its *size*:

In [ ]:
x.size

The *shape* of a tensor lists its length along each axis. The shape of a
vector has one entry, which equals the size:

In [ ]:
x.shape

the figure shows tensors with zero to three axes. A single
number, a *scalar*, is a tensor with no axes; its shape is empty. The size
of every tensor is the product of the entries of its shape. For the minibatch
of images, whose shape is $(256, 1, 28, 28)$, this is the count of 200,704
from the opening, since the single channel contributes a factor of 1.

![Tensors with zero, one, two, and three axes. The shape lists the length of each axis, and the size is the product of these lengths: 1, 4, 12, and 24 elements.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-rank-ladder.svg)

### Data Types

Every tensor also has a *data type*, its `dtype`, which fixes how each
element is stored. Deep learning computes mostly with 32-bit floating-point
numbers, `float32`, which take 4 bytes each and keep about 7 significant
decimal digits. That is fewer digits than a pocket calculator shows, and it
turns out to be enough. Other data need other types. The pixels of an image
are 8-bit unsigned integers, `uint8`, which take one byte each and hold the
integers from 0 to 255. The data type determines how much memory a tensor
needs: the minibatch of images takes 200,704 bytes as `uint8` and four times
as much, about 0.8 MB, as `float32`. Large models also compute with 16-bit
floating-point numbers, which halve the memory again at some cost in
precision (that section).

With integer arguments, `jnp.arange` creates an array of 32-bit integers,
hence the `float32` in the `arange` cell above. The method `astype` converts
an array to another type, as in `x.astype(jnp.int32)`.

### Initial Values

A model needs tensors with initial values before training can change them.
The simplest choice is a constant: the functions `zeros` and `ones` create a
tensor of a given shape filled with 0s or with 1s. Here is a tensor of zeros
with the shape $(2, 3, 4)$ of the figure:

In [ ]:
jnp.zeros((2, 3, 4))

Most parameters of a model, however, must start at small random values.
The reason has to wait until that section; for now we
only need to know how to draw them. The following tensor holds numbers
drawn independently from the standard normal distribution, a bell-shaped
distribution centered at 0 that places about 95% of its draws between $-2$
and $2$:

In [ ]:
# A key determines the sample: the same key always gives the same numbers
key = jax.random.key(random.getrandbits(32))  # A new seed on each run
jax.random.normal(key, (3, 4))

Random numbers differ from one run to the next, which is a nuisance when we
hunt for a bug or want others to reproduce a result. The numbers are, of
course, not random at all: a generator computes them from a starting value,
the *seed*, and the same seed gives the same numbers. Below, `a` and `c`
are the first draws from the same seed and agree, whereas `b` is a
different draw:

JAX has no global generator. Every draw takes an explicit *key*, and
`jax.random.split` derives new, independent keys from an existing one.

In [ ]:
key_a, key_b = jax.random.split(jax.random.key(42))
a = jax.random.normal(key_a, (3,))
b = jax.random.normal(key_b, (3,))  # A different key
c = jax.random.normal(key_a, (3,))  # Same key: c repeats a
a, b, c

Most experiments in this part of the book deliberately do not fix the seed.
Every training run starts from freshly drawn weights and visits its data in
a new random order, so its result is one draw from a distribution, and the
numbers on your screen will differ a little from those printed here. This is why we
report many results as a range, such as "between 82% and 84%": it is the
spread that we saw when we repeated the experiment. A single number with
four digits would claim more than one run can know. Our conclusions are
meant to hold anywhere in such a range, and yours should too.

Finally, we can write down the elements of a tensor ourselves, as nested
Python lists in which the outermost list runs along axis 0:

In [ ]:
jnp.array([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

This is all we need to build the running example. Until we load a real
dataset, random integers from 0 to 255 stand in for the pixels. Such images
show static rather than shirts, but only the shape and the type of the
minibatch matter here:

In [ ]:
# 256 grayscale images of 28 x 28 pixels, each pixel an integer in [0, 255]
key, subkey = jax.random.split(key)
images = jax.random.randint(subkey, (256, 1, 28, 28), 0, 256,
                            dtype=jnp.uint8)
images.shape, images.dtype

The shape reads: 256 images, 1 channel, 28 rows, and 28 columns.

## Reshaping and Indexing

### Reshaping

The same numbers often need a different arrangement. A classifier that
treats each image as a list of 784 pixel values needs the minibatch as a
matrix with one image per row, of shape $(256, 784)$. The function `reshape`
changes the shape of a tensor but neither its size nor the order of its
elements. On a small example, we turn the vector `x` of shape $(12,)$ into a
matrix `X` of shape $(3, 4)$. The elements fill the matrix one row at a
time, so `x[3]` becomes `X[0, 3]` (the figure):

![Reshaping a vector of 12 elements into a $3 \times 4$ matrix. The elements keep their order and fill the matrix one row at a time.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-reshape.svg)

In [ ]:
X = x.reshape(3, 4)
X

Since the size is fixed, one entry of the target shape follows from the
others: 12 elements in 3 rows make 4 columns. Writing $-1$ for one entry
lets the library infer it, so `x.reshape(-1, 4)` and `x.reshape(3, -1)` give
the same matrix as `x.reshape(3, 4)`. In the same way, we flatten every
image of the minibatch into a row of 784 pixels without writing out 784:

In [ ]:
images.reshape(256, -1).shape

Reshaping can also add or remove an axis of length 1, which leaves the
elements and their order unchanged. The first image of the minibatch,
`images[0]`, has shape $(1, 28, 28)$. A model that expects minibatches needs
it as a minibatch of one image, of shape $(1, 1, 28, 28)$, while a plot
needs a $28 \times 28$ matrix without the channel axis. The index `None`
inserts an axis of length 1, and `squeeze` removes one:

In [ ]:
image = images[0]
image[None].shape, image.squeeze(0).shape

### Indexing and Slicing

As with Python lists, we select elements of a tensor by *indexing*. Indices
start at 0, and negative indices count from the end. A *slice* `start:stop`
selects the positions from `start` up to, but not including, `stop`, and a
colon alone selects all positions. Indices for several axes are separated by
commas, and an index given alone applies to axis 0. Thus `X[-1]` selects the
last row of `X`, and `X[1:3]` selects the second and third rows:

In [ ]:
X[-1], X[1:3]

Note that the two results differ in shape. An integer index removes its
axis, so `X[-1]` is a vector, whereas a slice keeps its axis, so `X[1:3]` is
a matrix with two rows (the figure). The distinction looks
pedantic, but many an error about mismatched shapes goes back to it.

![Indexing a $3 \times 4$ matrix. The integer index `X[-1]` selects the last row and removes axis 0, leaving shape $(4,)$. The slice `X[1:3]` selects rows 1 and 2 and keeps axis 0, leaving shape $(2, 4)$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-index-read.svg)

On the minibatch, `images[0]` is the first image, `images[:32]` a smaller
minibatch of the first 32 images, and `images[:, 0, 14, 14]` the pixel in row 14 and
column 14 of every image:

In [ ]:
images[0].shape, images[:32].shape, images[:, 0, 14, 14].shape

An index can also be a tensor. Two tensors of integer positions, one for the
rows and one for the columns, select one element for each pair of
positions. This picks one entry from every row, as when we read off the
score that a classifier gives to the correct category of each image. The
matrix `X` holds the numbers 0 to 11 row by row, so the rows $(0, 1, 2)$ and
the columns $(3, 0, 2)$ read `X[0, 3]`, `X[1, 0]`, and `X[2, 2]`, which are
3, 4, and 10:

In [ ]:
rows, cols = jnp.arange(3), jnp.array([3, 0, 2])
X[rows, cols]

A tensor of booleans with the shape of `X`, a *mask*, selects the elements
of `X` where it is `True`. The comparison `X > 2` compares every element of
`X` with 2 and gives such a mask, and the operator `&` combines two masks
elementwise into one that is `True` where both are (*and*); `|` gives *or*.
The selected elements need not form rows and columns, so the result is a
vector:

The length of the result depends on the values in the mask, so indexing
with a mask works only outside functions compiled with `jax.jit`. The
function `where` below gives a result of fixed shape and works everywhere.

In [ ]:
mask = (X > 2) & (X < 8)
X[mask]

To keep the shape instead, `where(mask, A, B)` takes the elements of `A`
where the mask is `True` and those of `B` elsewhere. A single number in
place of `A` or `B` stands for that number at every position, so the
following sets every element outside the mask to 0:

In [ ]:
jnp.where(mask, X, 0)

### Writing Elements

So far we have only read elements. Can we also change them? Here the
libraries part ways, by design. MXNet and PyTorch tensors are *mutable*: an
assignment overwrites their elements in place. TensorFlow and JAX tensors
are *immutable*: once created, they never change.

Writing raises a question that reading could not. When we reshaped `x` into
`X`, did the library copy the elements? It need not have: the elements of
`x` lie in memory one after another, and so do the rows of `X`, so recording
a new shape for the same memory suffices. A tensor that shares its memory
with another in this way is called a *view*, and a write through either name
is then seen through both. For immutable tensors the question is idle. For
mutable ones we had better know the answer.

In place of an assignment, `X.at[1, 2].set(17)` returns a new array that
agrees with `X` except at position $(1, 2)$, and leaves `X` unchanged:

In [ ]:
X_new = X.at[1, 2].set(17)  # X is unchanged
X_new

To give many elements the same value, we index with a slice. Here
`[:2, :]` selects the first two rows, with the colon taking all columns:

In [ ]:
X_new = X_new.at[:2, :].set(12)
X_new

## Operations

### Elementwise Operations

An operation is *elementwise* if it applies a function of numbers to each
element of a tensor separately, or to each pair of elements at the same
position in two tensors of the same shape. The arithmetic operators `+`,
`-`, `*`, `/`, and `**` (power) all work this way. We try them on two new
vectors, reusing the names `x` and `y`:

In [ ]:
x = jnp.array([1.0, 2, 4, 8])
y = jnp.array([2.0, 2, 2, 2])
x + y, x - y, x * y, x / y, x ** y

Each result has the shape of the operands, and each element depends only on
the two elements at its position: `x / y` halves every element of `x`, and
`x ** y` squares it. In symbols, a function $f$ of two numbers turns two
tensors $\mathbf{u}$ and $\mathbf{v}$ of the same shape into the tensor
$\mathbf{c}$ with elements $c_i = f(u_i, v_i)$, where $i$ runs over all
positions.

Functions of a single number, such as $e^x$, apply to each element in the
same way:

In [ ]:
jnp.exp(x)

We are now in a position to carry out the rescaling from the beginning of
the section. Dividing the minibatch by 255 divides every pixel and returns a
new tensor. Division with `/` always returns floating-point numbers, here
`float32`, even where a quotient happens to be an integer. (Addition makes
no such promise: the sum of two `uint8` tensors is again `uint8`, with
consequences that an exercise explores.) Strictly speaking, the definition
above does not cover `images / 255`, since the second operand is a single
number rather than a tensor of the same shape. We return to this case in
the next subsection:

In [ ]:
scaled = images / 255
print(scaled.dtype, images[0, 0, 0, :4], scaled[0, 0, 0, :4], sep='\n')

The type has changed from `uint8` to `float32`, and the first four pixels of
the first image, integers from 0 to 255, have become numbers between 0
and 1.

Comparisons, such as `X > 2` above, are elementwise too. They return a
tensor of booleans that is `True` where the comparison holds:

In [ ]:
x == y

Not every operation is elementwise. Summing all the elements of a tensor
gives a tensor with a single element. Summing a tensor of booleans counts
its `True` entries, because `True` counts as 1 and `False` as 0; this is how
we will count the correct predictions of a classifier:

In [ ]:
x.sum(), (x == y).sum()

A sum can also run along a single axis and leave, for instance, one sum per
row of a matrix. Such reductions along an axis are the subject of
that section.

### Broadcasting

Elementwise operations are simple when the two tensors have the same shape.
Unfortunately, they rarely do. Model code constantly combines a minibatch,
stored with one example per row, with a vector that holds one number per
column: preprocessing, for instance, subtracts the same vector of column
means from every example. Copying the vector once for every example, only to
match the shape of the minibatch, would waste memory and time.
*Broadcasting*, a convention that deep learning libraries adopted from
NumPy, applies the vector to every row without copying it. Here we add the
vector $(100, 200, 300, 400)$ to each row of a new $3 \times 4$ matrix `X`,
which again holds the numbers 0 to 11:

In [ ]:
X = jnp.arange(12, dtype=jnp.float32).reshape(3, 4)
v = jnp.array([100.0, 200, 300, 400])
X + v

Every row received the same four offsets, one per column.

Broadcasting follows a simple rule, and it is worth knowing by heart. Line
the two shapes up from the *right*
and compare them axis by axis: two axes are compatible when they are equal
or when one of them is $1$ (a missing leading axis counts as $1$). Each axis
of length $1$ is then *stretched* to match the other, by virtually copying
its entries, and the elementwise operation is applied to the results. If any
pair of axes is incompatible, the operation raises an error rather than
guessing. For `X + v`, the shapes $(3, 4)$ and $(4,)$ line up as $(3, 4)$
and $(1, 4)$, so the single row of `v` is stretched to three rows. The
division of the minibatch by 255 was the simplest case: a scalar has no axes,
so it is stretched along all of them.

Both operands can be stretched at once. In
the figure, a column of shape $(3, 1)$ and a row of
shape $(1, 2)$ are each stretched to shape $(3, 2)$:

![Broadcasting stretches axes of length 1. The column `a` of shape $(3, 1)$ and the row `b` of shape $(1, 2)$ are both expanded to shape $(3, 2)$ (dashed: virtual copies) before the elementwise addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ndarray-broadcasting.svg)

In [ ]:
a = jnp.arange(3).reshape(3, 1)
b = jnp.arange(2).reshape(1, 2)
a, b

Their sum adds every element of the column to every element of the row:

In [ ]:
a + b

What if the shapes are not compatible? Lining up $(3, 2)$ and
$(2, 3)$ from the right pairs $2$ with $3$ and $3$ with $2$; in both pairs
the lengths differ and neither is $1$, so the library refuses:

In [ ]:
try:
    jnp.ones((3, 2)) + jnp.ones((2, 3))
except Exception as e:
    print(e)

The rule also predicts what happens on the minibatch of images. Subtracting a
mean image of shape $(28, 28)$ from the minibatch of shape $(256, 1, 28, 28)$
works: the last two axes agree, and the mean image is stretched along the first
two. Adding a vector of 256 numbers, one per image, to the flattened minibatch
of shape $(256, 784)$ fails, because lining up the shapes from the right pairs
256 with 784. Reshaping the vector to shape $(256, 1)$ repairs this, since
$(256, 1)$ and $(256, 784)$ are compatible.

### Concatenation

Broadcasting combines tensors position by position. *Concatenation* instead
joins tensors end to end along an existing axis. Joining two minibatches of
256 images along axis 0 gives one minibatch of 512 images;
joining two sets of features for the same examples along axis 1 gives every
example more features. The lengths along all other axes must agree. We
concatenate two $3 \times 4$ matrices, `X` (defined afresh) and `Y`, in both
ways. Along axis 0, the result has $3 + 3 = 6$ rows; along axis 1, it has
$4 + 4 = 8$ columns:

In [ ]:
X = jnp.arange(12, dtype=jnp.float32).reshape(3, 4)
Y = jnp.array([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
jnp.concatenate((X, Y), axis=0), jnp.concatenate((X, Y), axis=1)

*Stacking* joins tensors of the same shape along a new axis instead. This is
how a minibatch arises from single examples: stacking 256 images of shape
$(1, 28, 28)$ gives the minibatch of shape $(256, 1, 28, 28)$. Stacking `X`
and `Y` along a new axis 0 gives shape $(2, 3, 4)$, and along a new last
axis, shape $(3, 4, 2)$:

In [ ]:
jnp.stack((X, Y)).shape, jnp.stack((X, Y), axis=2).shape

## Memory and Conversion

### Allocation and In-Place Updates

Most operations return a new tensor, which needs new memory. The statement
`Y = Y + X` computes the sum in newly allocated memory and then binds the
name `Y` to the new tensor. Python's `id` function, which returns a number
that identifies an object for as long as the object exists, shows that `Y`
now names a different object:

In [ ]:
before = id(Y)
Y = Y + X
id(Y) == before

For small tensors this costs little, and we can forget about it. A model
with a billion parameters, however, needs 4 GB to store them as `float32`
numbers, and training updates all of them at every step. If every update
allocated new memory for its result, the program would need room for the old
and the new values at the same time, and would spend time on the
allocations. An *in-place* update writes the result into the memory that the
tensor already occupies. It has a second effect, which is easily overlooked
and matters just as much: it reaches every part of the program that refers
to the tensor, whereas rebinding one name leaves other names pointing at the
old values. In-place updates come with one restriction, whose reason will
become clear in that section: a library that records operations to
compute derivatives may still need an old value, and then an in-place update
must not overwrite it.

JAX arrays are immutable, so JAX has no in-place updates: `Y + X` and
`X.at[...].set(...)` always produce a new array, and the memory of an old
array is freed once no name refers to it. Avoiding the extra copy is the
job of JAX's compiler: training code compiles its update step, and the
compiled step can reuse memory that is no longer needed.

In particular, `X += Y` is shorthand for `X = X + Y`, so it binds `X` to a
new array:

In [ ]:
before = id(X)
X += Y
id(X) == before

### Conversion to Other Python Objects

Plotting, pandas, and many scientific libraries take NumPy arrays or Python
numbers rather than tensors, so we convert tensors at the boundary between
our code and theirs. Whether a conversion copies the data or shares the
memory, as a view does, depends on the library.

The function `jax.device_get` returns a JAX array as a NumPy array, and
`jax.device_put` converts it back. For an array on an accelerator,
`device_get` copies the data to main memory. For an array that is already in
main memory, the NumPy array may share its memory with the JAX array.
Either way, the NumPy array is read-only, so the JAX array stays immutable.

In [ ]:
A = jax.device_get(X)
B = jax.device_put(A)
type(A), type(B)

Writing an element of the NumPy array `A` fails, because `A` is read-only:

In [ ]:
try:
    A[0, 0] = 100
except ValueError as e:
    print(e)

A tensor with a single element converts to a Python number with the method
`item` or with the built-in functions `float` and `int`. Printing a summary
number such as an average error, or comparing it with a threshold in Python,
needs such a conversion. It is not free, though: when the computation runs
on a GPU, the conversion has to wait for it to finish, so training code
converts as rarely as possible.

In [ ]:
a = jnp.array(3.5)
a, a.item(), float(a), int(a)

The numbers in this section were made up. Real data arrive in files, often
as tables with missing entries and with text in some of the columns, and
they are rarely as tidy as the output of `arange`. The next section turns
such a table into tensors.

## Summary

Deep learning code stores its data, parameters, and intermediate results in
tensors, arrays with a shape and a data type. It computes with whole tensors
rather than with loops over their elements: one expression divides all
200,704 pixels of a minibatch by 255. Reshaping, indexing, and
concatenation rearrange, select, and join elements. Elementwise operations
apply a function of numbers at every position, and broadcasting combines
tensors of different shapes by lining up their shapes from the right and
stretching axes of length 1. Most operations allocate a new tensor; where
memory matters, libraries with mutable tensors update them in place, and
libraries with immutable tensors reuse memory through mutable variables or
through their compiler.

## Exercises

1. [code] **Inferred reshape axis.** A minibatch of $n$ grayscale images of
   $28 \times 28$ pixels arrives as a tensor of shape $(n, 1, 28, 28)$.
   Each image must be flattened into a row of 784 values. Check every
   prediction below in code.
    1. Predict the result of reshaping the minibatch to $(-1, 784)$ and to
       $(256, -1)$, once for a full minibatch of $n = 256$ images and once
       for a final minibatch of $n = 100$ images. State which of the two
       targets works for every minibatch size.
    1. Suppose the images were loaded with three color channels by
       mistake, so that the minibatch has shape $(n, 3, 28, 28)$. Predict the
       result of the reshape to $(-1, 784)$, and state whether this step
       detects the mistake.

1. **Shifted slices.** A tensor `tokens` of shape $(4, 11)$ holds four
   sequences of *tokens*, integers that stand for words, one sequence per
   row. A model reads the first 10 tokens of each row and predicts, at
   every position, the token that follows. Write slice
   expressions for the input tensor and the target tensor, each of shape
   $(4, 10)$, so that the target at position $t$ is the token that
   follows the input at position $t$. Then give expressions for both
   tensors that stay correct without change when each row holds $T + 1$
   tokens and the model reads the first $T$, for every $T \geq 1$.

1. [code] **Broadcasting shapes.** Broadcasting aligns two shapes from the
   right.
    1. Replace `a` and `b` with two tensors that each have three axes,
       whose shapes differ on every axis and that still broadcast. Predict the shape of `a + b`,
       then run the code and check your prediction.
    1. A model returns predictions `y_hat` of shape $(5, 1)$ for five
       examples whose labels `y` form a vector of shape $(5,)$. Predict the
       shape of `y_hat - y`. Then let `y` hold the labels $0, 1, 2, 3, 4$
       as floating-point numbers and set `y_hat` to `y` reshaped to
       $(5, 1)$, so that every prediction equals its label. Compute the
       mean of the squared entries of `y_hat - y`, explain the value, and
       give the reshape of `y` that pairs each prediction with its own
       label.

1. [code] **Limits of data types.** The pixels of `images` are stored as
   `uint8`, and the rest of the book computes in `float32`.
    1. Predict the result of adding 100 to a `uint8` tensor that holds the
       value 200, and check your prediction. Explain the result, and
       brighten the first image of the minibatch by 100 intensity levels
       without this error by converting its type first.
    1. A `float32` number stores 24 significant binary digits. Explain why
       `float32` represents every integer up to $2^{24} = 16{,}777{,}216$,
       and check that adding 1 to $2^{24}$ in `float32` leaves it
       unchanged. A training loop counts the examples it has processed in a
       `float32` tensor. After how many examples does the count stop
       growing, and which data type should the counter have instead?

1. [code] **Masks on the minibatch.** The pixels of `images` are random
   integers from 0 to 255, so about half of them exceed 127.
    1. With one comparison and one sum, count the pixels of the minibatch
       that exceed 127, and compare the count with half of the size of
       `images`. Explain why summing a mask counts.
    1. Use `where` to make a thresholded copy of `images` in which every
       pixel above 127 becomes 255 and every other pixel 0. Confirm that the
       copy has the shape of `images` and holds only these two values.
    1. `images[images > 127]` starts from the same mask. Predict its shape,
       check, and explain why a mask used as an index must return a tensor
       with one axis, whatever the shape of `images`, while `where` keeps
       the shape. Name a task for which each form is the right one. Then
       select the bright pixels of the first image alone, with a slice and
       a mask.

6. [code] **In-place composition.** ● JAX arrays are immutable, so every
   eager operation allocates a new array. Write `(A + B) * (-A / 2)` as a
   function `f(A, B)` and count the intermediate arrays that eager
   evaluation creates. The function `jax.jit(f)` compiles `f` into one optimized
   program; inspect that program with
   `jax.jit(f).lower(A, B).compile().as_text()` and report how many fused
   computations remain. An in-place implementation, as in NumPy, writes
   the result into the memory of `A` and allocates no new array. State
   which of the eager allocations the compiled program removes, and
   whether it still allocates an array that the in-place implementation
   avoids.

    *Adapted from Nicolas P. Rougier,
    [100 numpy exercises](https://github.com/rougier/numpy-100/blob/master/100_Numpy_exercises.md),
    problem 35.*

7. [code] **Loops against broadcasting.** ● For a vector `a` of 200
   elements and a vector `b` of 300 elements, compute the table of all
   absolute differences $d_{ij} = |a_i - b_j|$, of shape $(200, 300)$.
    1. Compute the table once with two nested Python loops, and once with
       a single broadcast expression after reshaping `a` to $(200, 1)$ and
       `b` to $(1, 300)$. Confirm that the two results agree, time both,
       and report the speedup.
    1. The broadcast expression copies neither input. How many bytes does
       its result take in `float32` when both vectors have 10,000
       elements? What does this say about the memory that broadcasting
       can create?

[Discussions](https://d2l.discourse.group/t/17966)